In [73]:
import re
import pandas as pd 
from collections import defaultdict
from typing import List, Dict, Tuple

In [74]:
train_text = """ I love Deep Learning. 
And I love NLP, with or without deep learning. 
In general I love AI, but I don't like A.B.C."""


In [75]:
def tokenize(text: str, pattern: str = None) -> List[str]:
    pattern = r"(?:\w+')\w+|(?:[A-Z]\.)+|\w+(?:-\w+)*|[\w+\.]"
    tokens = re.findall(pattern, text)
    return tokens


In [76]:
train_tokens = tokenize(train_text)
# train_tokens

In [77]:

def ngram_counts(tokens: List[str], n: int) -> Dict[Tuple[str, ...], Dict[str, int]]:
    """ 
    Compute the ngram counts of given tokens with specifed context size.
    
    Args:
        tokens: list of tokenized strings.
        n size of context, n=1 >> bigram, n=2 >> trigram etc. 

    Returns:
        Nested Dict of (context, next_word) counts.
    """
    counts = defaultdict(lambda: defaultdict(int))
    for i in range(len(tokens) - n):
        context = tuple(tokens[i: i + n])
        next_ = tokens[i+n]
        counts[context][next_] += 1
    return counts
    
    

In [78]:
unicnt = ngram_counts(train_tokens, n=0)
bicnt = ngram_counts(train_tokens, n=1)
tricnt = ngram_counts(train_tokens, n=2)


In [80]:
tricnt_df = pd.DataFrame.from_dict(tricnt, orient="index").fillna(0)
tricnt_df.T

,I,love,Deep,deep,Learning,.,In,AI,And,general,...,NLP,with,or,without,learning,.,love,but,I,don't
,love,Deep,Learning,learning,.,And,general,but,I,I,...,with,or,without,deep,.,In,AI,I,don't,like
Deep,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NLP,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AI,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Learning,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
.,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
And,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
I,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
love,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
with,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [69]:
def coocurr(tokens: List[str], window_size: int) -> Dict[str, Dict[str, int]]:
    """ 
    Compute the co-occurrence counts of given tokens with specifed context (windon) size.
    
    Args:
        tokens: list of tokenized strings.
        window_size: size of context (window)

    Returns:
        Nested Dict of (context, next_word) counts.
    """
    counts = defaultdict(lambda: defaultdict(int))
    for i in range(len(tokens)):
        start = max(0, i-window_size)
        end = min(len(tokens), i + window_size + 1)
        for j in range(start, end):
            if i != j:
                counts[tokens[i]][tokens[j]] += 1
    return counts
        

In [72]:
cooc = coocurr(train_tokens, window_size=4)
cooc_df = pd.DataFrame.from_dict(cooc, orient="index").fillna(0)
cooc_df

    


,love,Deep,Learning,.,And,NLP,with,or,learning,In,general,AI,but,I,don't,like,A.B.C.,without,deep
I,4.0,2.0,2.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0,1.0,1.0,0.0,0.0
Deep,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
Learning,2.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
.,3.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,1.0,1.0
And,2.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
NLP,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
with,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
or,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
without,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
In,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
